# 02 - Data Validation

## 1. Purpose
Run the same data quality checks defined in `sql/01_data_quality/` in Python,
so the SQL layer and the Python layer can be independently cross-checked
against each other (reconciliation). No rows are modified in this notebook.


## 2. Imports

In [1]:
import pandas as pd
import numpy as np


## 3. Data Loading

In [1]:
df = pd.read_csv("../data/raw/monthly_revenue_raw.csv")
df['sales_month_parsed'] = pd.to_datetime(df['sales_month'], format='%Y-%m', errors='coerce')
df.head()


  sales_month  current_month_revenue sales_month_parsed
0     2016-10                  43000         2016-10-01
1     2016-11                 105000         2016-11-01
2     2016-12                 253000         2016-12-01
3     2017-01                 390000         2017-01-01
4     2017-02                 362000         2017-02-01

## 4. Analysis

In [1]:
# --- Null checks ---
print("Null counts:")
print(df[['sales_month', 'current_month_revenue']].isnull().sum())


Null counts:
sales_month              0
current_month_revenue    0
dtype: int64


In [1]:
# --- Duplicate month checks ---
dupes = df['sales_month'].duplicated().sum()
print(f"Duplicate sales_month rows: {dupes}")


Duplicate sales_month rows: 0


In [1]:
# --- Date parse / continuity checks ---
unparseable = df['sales_month_parsed'].isnull().sum()
print(f"Unparseable dates: {unparseable}")

full_range = pd.period_range(
    start=df['sales_month_parsed'].min(),
    end=df['sales_month_parsed'].max(),
    freq='M'
)
present = pd.PeriodIndex(df['sales_month_parsed'], freq='M')
missing_months = sorted(set(full_range) - set(present))
print(f"Expected months: {len(full_range)} | Present: {len(present)} | Missing: {missing_months}")


Unparseable dates: 0
Expected months: 23 | Present: 23 | Missing: []


In [1]:
# --- Revenue validation ---
neg_or_zero = (df['current_month_revenue'] <= 0).sum()
print(f"Negative or zero revenue rows: {neg_or_zero}")
print(df['current_month_revenue'].describe())

mean = df['current_month_revenue'].mean()
std = df['current_month_revenue'].std()
z = (df['current_month_revenue'] - mean) / std
outliers = df[z.abs() > 3]
print(f"\nRows beyond 3 std dev: {len(outliers)}")


Negative or zero revenue rows: 0
count          23.00
mean      684,239.17
std       315,444.74
min        43,000.00
25%       489,500.00
50%       680,000.00
75%     1,005,987.00
max     1,061,000.00
Name: current_month_revenue, dtype: float64

Rows beyond 3 std dev: 0


## 5. Findings

All checks below match the SQL results in `sql/01_data_quality/` exactly:

| Check | Result |
|---|---|
| Null values | 0 |
| Duplicate months | 0 |
| Unparseable dates | 0 |
| Missing months in sequence | 0 (continuous Oct-2016 to Aug-2018) |
| Negative/zero revenue | 0 |
| Values beyond 3 standard deviations | 0 |

**This dataset has no data quality defects.** That is itself worth stating
explicitly rather than assuming -- a clean 23-row monthly series is a
reasonable, auditable starting point for the growth analysis that follows.

The one real limitation is **coverage**, not quality: 23 months is enough for
MoM analysis but only supports 11 months of trailing Year-over-Year
comparison (see `docs/metric_definitions.md`), and the first ~2-3 months sit
in a near-zero-revenue ramp period that produces mathematically extreme
(but not "wrong") growth percentages.


## 6. Conclusion

Data is validated and clean. Proceed to `03_revenue_eda.ipynb` for
exploratory analysis of the revenue trend, distribution, and volatility.
